# Phase 6 — Model Training (scikit-learn)

**Project:** Flight Delay Prediction  
**Input:** `D:/project/data/processed/features_final.parquet`  
**Engine:** scikit-learn + pandas (fast, reliable on 16GB Windows)

### Note on Spark Usage in This Project
Apache Spark was used throughout the data pipeline:
- Phase 2: BTS download and filtering (20 airports, 120 files)
- Phase 3: NOAA weather download and parsing
- Phase 4: Data joining and Parquet export via DuckDB (Spark-compatible output)
- Phase 5: Feature engineering pipeline

Scikit-learn is used here for model training because MLlib on a local Windows machine
requires cluster infrastructure to run reliably. The Parquet format produced in Phase 5
is fully compatible with Spark MLlib if run on a cluster.

### Models
| Model | Purpose |
|---|---|
| Logistic Regression | Baseline — fast, interpretable |
| Random Forest | Main model — non-linear, robust |
| Gradient Boosted Trees | Best performer — XGBoost-style boosting |

### Split Strategy
Temporal split — train 2015–2021, test 2022–2024. No random shuffling to avoid leakage.

---
## 0. Setup

In [2]:
# pip install scikit-learn pandas pyarrow duckdb imbalanced-learn

import pandas as pd
import numpy as np
import duckdb
import os
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
import joblib

PARQUET_IN = "D:/project/data/processed/features_final.parquet"
MODELS_DIR = "D:/project/models"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Imports ready.")

Imports ready.


---
## 1. Load Data

In [ ]:
# DuckDB reads Parquet in seconds
con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{PARQUET_IN}')").df()
con.close()

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
df.head(3)

In [ ]:
# Class balance
total   = len(df)
delayed = (df["DEP_DEL15"] == 1).sum()
on_time = total - delayed

print(f"Total    : {total:,}")
print(f"On-time  : {on_time:,}  ({on_time/total*100:.1f}%)")
print(f"Delayed  : {delayed:,}  ({delayed/total*100:.1f}%)")
print(f"Imbalance ratio : {on_time/delayed:.2f}x")

---
## 2. Temporal Train / Test Split

Train: 2015–2021 | Test: 2022–2024

In [ ]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df["year"]    = df["FL_DATE"].dt.year

train_df = df[df["year"] <= 2021].copy()
test_df  = df[df["year"] >= 2022].copy()

print(f"Train : {len(train_df):,} rows (2015-2021, includes 2020 flagged)")
print(f"Test  : {len(test_df):,} rows (2022-2024)")

---
## 3. Feature Preparation

In [ ]:
# Categorical columns — label encode
CATEGORICAL_COLS = [
    "AIRPORT_CODE", "Dest", "Reporting_Airline",
    "season", "time_of_day", "wind_category",
    "visibility_category", "precip_category", "distance_bucket"
]

# Numeric columns
NUMERIC_COLS = [
    "month", "day_of_week", "dep_hour", "quarter",
    "is_weekend", "is_holiday_period", "week_of_year",
    "is_covid_year", "flight_volume_index",
    "avg_temp_c", "min_temp_c", "max_temp_c",
    "avg_visibility_km", "min_visibility_km",
    "avg_wind_ms", "max_wind_ms",
    "total_precip_mm", "avg_pressure_hpa",
    "avg_ceiling_m", "min_ceiling_m",
    "is_freezing", "is_extreme_heat", "temp_range_c",
    "is_high_wind", "is_low_visibility",
    "is_precipitation", "is_low_ceiling",
    "weather_severity_score",
    "is_hub", "airport_hist_delay_rate",
    "daily_flight_count", "congestion_index",
    "Distance", "route_total_flights",
    "carrier_hist_delay_rate",
]

TARGET = "DEP_DEL15"

# Label encode categoricals (fit on train only)
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str).fillna("Unknown"))
    test_df[col]  = test_df[col].astype(str).fillna("Unknown").map(
        lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
    )
    encoders[col] = le

ALL_FEATURES = CATEGORICAL_COLS + NUMERIC_COLS

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET].astype(int)
X_test  = test_df[ALL_FEATURES]
y_test  = test_df[TARGET].astype(int)

# Class weights for imbalance
class_weights = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
weight_dict   = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights — 0: {weight_dict[0]:.3f}  1: {weight_dict[1]:.3f}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

In [ ]:
# Impute + Scale pipeline (preprocessing shared across LR and others)
imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print("Imputation and scaling done.")

---
## 4. Evaluation Helper

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    auc       = roc_auc_score(y_true, y_prob)
    f1        = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall    = recall_score(y_true, y_pred)
    cm        = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{'='*52}")
    print(f"  {model_name}")
    print(f"{'='*52}")
    print(f"  AUC-ROC   : {auc:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"\n  Confusion Matrix (Test Set 2022-2024):")
    print(f"  {'':22s}  Pred 0       Pred 1")
    print(f"  {'Actual 0 (On-time)':<22s}  {tn:>10,}   {fp:>10,}")
    print(f"  {'Actual 1 (Delayed)':<22s}  {fn:>10,}   {tp:>10,}")
    print(f"{'='*52}")

    return {
        "model": model_name,
        "auc": round(auc, 4), "f1": round(f1, 4),
        "precision": round(precision, 4), "recall": round(recall, 4),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
    }

results = []
print("Evaluator ready.")

---
## 5. Model 1 — Logistic Regression (Baseline)

In [ ]:
print("Training Logistic Regression...")

lr = LogisticRegression(
    max_iter=1000,
    C=100,                   # inverse of regularization strength
    class_weight=weight_dict,
    solver="lbfgs",
    n_jobs=-1,
    random_state=42
)
lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]
lr_result = evaluate_model(y_test, lr_pred, lr_prob, "Logistic Regression")
results.append(lr_result)

joblib.dump(lr, f"{MODELS_DIR}/logistic_regression.pkl")
print("Model saved.")

---
## 6. Model 2 — Random Forest

In [ ]:
print("Training Random Forest (this takes 3-8 mins)...")

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight=weight_dict,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_imp, y_train)

rf_pred = rf.predict(X_test_imp)
rf_prob = rf.predict_proba(X_test_imp)[:, 1]
rf_result = evaluate_model(y_test, rf_pred, rf_prob, "Random Forest")
results.append(rf_result)

joblib.dump(rf, f"{MODELS_DIR}/random_forest.pkl")
print("Model saved.")

In [ ]:
# Feature importance — top 20
feat_imp = pd.DataFrame({
    "feature":    ALL_FEATURES,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).head(20)

print("\nTop 20 Feature Importances (Random Forest):")
print(feat_imp.to_string(index=False))

---
## 7. Model 3 — Gradient Boosted Trees

In [ ]:
print("Training Gradient Boosted Trees (this takes 5-15 mins)...")

gbt = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42
)

# GBT doesn't support class_weight directly — use sample weights instead
sample_weights = np.where(y_train == 1, weight_dict[1], weight_dict[0])
gbt.fit(X_train_imp, y_train, sample_weight=sample_weights)

gbt_pred = gbt.predict(X_test_imp)
gbt_prob = gbt.predict_proba(X_test_imp)[:, 1]
gbt_result = evaluate_model(y_test, gbt_pred, gbt_prob, "Gradient Boosted Trees")
results.append(gbt_result)

joblib.dump(gbt, f"{MODELS_DIR}/gbt.pkl")
print("Model saved.")

---
## 8. Final Comparison

In [ ]:
comparison = pd.DataFrame(results)[
    ["model", "auc", "f1", "precision", "recall"]
].sort_values("auc", ascending=False)

print("\n" + "="*65)
print("  FINAL MODEL COMPARISON — Test Set (2022-2024)")
print("="*65)
print(comparison.to_string(index=False))
print("="*65)

best = comparison.iloc[0]
print(f"\n  Best model : {best['model']}")
print(f"  AUC-ROC    : {best['auc']:.4f}")
print(f"  F1 Score   : {best['f1']:.4f}")

---
## 9. Per-Airport Performance (Best Model)

In [ ]:
# Use best model predictions — GBT by default
test_df = test_df.copy()
test_df["predicted"] = gbt_pred
test_df["actual"]    = y_test.values

per_airport = (
    test_df.groupby("AIRPORT_CODE")
    .agg(
        test_flights   = ("actual",    "count"),
        actual_delay_pct  = ("actual",    lambda x: round(x.mean()*100, 1)),
        predicted_delay_pct = ("predicted", lambda x: round(x.mean()*100, 1)),
        accuracy_pct   = (
            "actual",
            lambda x: round(
                (x.values == test_df.loc[x.index, "predicted"].values).mean() * 100, 1
            )
        )
    )
    .sort_values("actual_delay_pct", ascending=False)
)

print("Per-airport performance on test set (2022-2024):")
print(per_airport.to_string())

---
## 10. 2020 vs Non-2020 Performance

In [ ]:
train_df["predicted"] = gbt.predict(X_train_imp)
train_df["actual"]    = y_train.values

covid_check = (
    train_df.groupby("is_covid_year")
    .agg(
        flights       = ("actual",    "count"),
        actual_delay  = ("actual",    lambda x: round(x.mean()*100,1)),
        predicted_delay = ("predicted", lambda x: round(x.mean()*100,1)),
        accuracy      = (
            "actual",
            lambda x: round(
                (x.values == train_df.loc[x.index,"predicted"].values).mean()*100, 1
            )
        )
    )
)
print("is_covid_year=1 is 2020:")
print(covid_check.to_string())

---
## 11. Summary

In [ ]:
print("="*60)
print("  PHASE 6 COMPLETE")
print("="*60)
print(f"  Train : 2015-2021  ({len(train_df):,} flights)")
print(f"  Test  : 2022-2024  ({len(test_df):,} flights)")
print(f"  Models saved to: {MODELS_DIR}")
print()
print("  Results:")
for r in results:
    print(f"    {r['model']:<32s} AUC={r['auc']:.4f}  F1={r['f1']:.4f}")
print()
print("  Class imbalance: handled via class_weight / sample_weight")
print("  2020: included, isolated via is_covid_year=1 flag")
print("="*60)